# Notebook 4: Hugging Face API & Service Health Check
Verifies the token and that each configured model responds.

In [ ]:
import os, sys
from dotenv import load_dotenv

# Run from the notebooks/ folder: project root is one level up.
sys.path.append('..')
load_dotenv('../.env')

from huggingface_hub import HfApi
from utils.config import Config

print("=== HUGGING FACE SERVICE HEALTH CHECK ===")
print(f"API Key Configured: {'YES' if Config.is_huggingface_configured() else 'NO (free Google speech fallback only; sign recognition off)'}")
print(f"STT Model:  {Config.HUGGINGFACE_STT_MODEL}")
print(f"TTS Model:  {Config.HUGGINGFACE_TTS_MODEL}")
print(f"Sign Model: {Config.HUGGINGFACE_SIGN_MODEL}")

In [ ]:
# 1) Is the token valid?
if Config.is_huggingface_configured():
    try:
        who = HfApi().whoami(token=Config.HUGGINGFACE_API_KEY)
        print(f"Token OK - account: {who.get('name')}")
    except Exception as e:
        print(f"Token check FAILED: {e}")
else:
    print("HF API key not set in .env - skipping token check.")

In [ ]:
# 2) Do the models exist on the Hub, and are they served by an inference provider?
import time
api = HfApi()
for label, model in [("STT", Config.HUGGINGFACE_STT_MODEL), ("TTS", Config.HUGGINGFACE_TTS_MODEL), ("Sign", Config.HUGGINGFACE_SIGN_MODEL)]:
    t = time.time()
    try:
        info = api.model_info(model, expand=["inferenceProviderMapping"])
        providers = getattr(info, "inference_provider_mapping", None)
        print(f"{label:5} {model}: exists | providers: {providers or 'none (serverless inference not available)'} | {time.time()-t:.2f}s")
    except Exception as e:
        print(f"{label:5} {model}: lookup failed - {e}")